<left>
    <img src="https://weclouddata.s3.amazonaws.com/images/logos/wcd_logo_new_2.png" width='20%'>
</left>

<h1 align="left"> Demo: Multi-Agent System Walkthrough </h1>
<center align="left"> <font size='4'>  Developed by: </font><font size='4' color='#33AAFBD'>WeCloudData</font></center>
<br>

**Purpose:**  
Present a complete, working **multi-agent system** built with LangGraph that integrates planning, retrieval (using FAISS), execution, and tracking.

**Scenario:** AI Research Report Generator  
The system generates a professional research report on **"Latest Advancements in Multimodal AI Models – 2026"** by coordinating multiple specialized agents.

**Key Features Demonstrated:**
- Multi-agent architecture using LangGraph
- FAISS vector retrieval from a custom sample dataset
- Clear communication traces
- Task decomposition, execution tracking, and coordination

---

## 1. Setup

In [1]:
# Run this cell first
!pip install -q "langchain==0.3.*" "langchain-openai==0.2.*" "langchain-community==0.3.*" "langgraph==0.2.*" faiss-cpu duckduckgo-search --force-reinstall

import os
from getpass import getpass

if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

print("Setup complete!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.4/109.4 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.2/94.2 kB 8.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.7/58.7 kB 4.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.4/40.4 kB 426.6 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 23.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 58.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.7/153.7 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 73.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 67.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.6

In [3]:
!pip install -U ddgs

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.6/70.6 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.7/161.7 kB 8.7 MB/s eta 0:00:00


## 2. Multi-Agent Architecture Overview

We are building a **Supervisor + Worker** architecture using LangGraph with **FAISS vector retrieval** from our own custom dataset.

- **Supervisor/Planner Agent**: Creates plan and coordinates
- **Researcher Agent**: Retrieves information from FAISS vector store
- **Calculator Agent**: Performs calculations
- **Reporter Agent**: Compiles the final report

In [4]:
from langchain_openai import ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings
from langchain_core.tools import tool
from langchain_community.tools import DuckDuckGoSearchRun
from langgraph.graph import StateGraph, END
from typing import TypedDict, Annotated, List
import operator

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
search_tool = DuckDuckGoSearchRun()

print("Core components loaded")

Core components loaded


## 3. Create Sample Dataset + FAISS Vector Store

In [5]:
# Custom Sample Dataset - Multimodal AI 2026
documents = [
    "GPT-5 by OpenAI (2026) supports text, image, video, audio, and 3D modalities with 1M context window.",
    "Gemini 2.5 by Google features real-time video understanding and advanced multimodal reasoning.",
    "Claude 4 by Anthropic excels in long-context document analysis and secure enterprise multimodal use.",
    "Llama 4 by Meta introduces native multimodal capabilities with open weights and 128K context.",
    "Average number of modalities in top multimodal models in 2026 is expected to be 4.5.",
    "Major challenge in multimodal AI is consistent alignment across different data types.",
    "New benchmark MMBench-2026 shows GPT-5 achieving 92.4% score on complex multimodal tasks."
]

embeddings = OpenAIEmbeddings()
vectorstore = FAISS.from_texts(documents, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

print("Custom dataset loaded into FAISS vector store")

Custom dataset loaded into FAISS vector store


## 4. Define LangGraph State and Nodes

In [6]:
class AgentState(TypedDict):
    messages: Annotated[List[str], operator.add]
    next: str
    plan: str
    research_data: str
    final_report: str

# Nodes
def planner_node(state: AgentState):
    prompt = f"Create a short 4-step plan for this task: {state['messages'][-1]}"
    plan = llm.invoke(prompt).content
    return {"plan": plan, "next": "researcher"}

def researcher_node(state: AgentState):
    query = "Latest multimodal AI models and advancements in 2026"
    docs = retriever.invoke(query)
    research = "\n".join([doc.page_content for doc in docs])
    return {"research_data": research, "next": "calculator"}

def calculator_node(state: AgentState):
    calc_prompt = f"Using this data: {state['research_data']}\nCalculate average modalities and summarize key stats."
    result = llm.invoke(calc_prompt).content
    return {"messages": [result], "next": "reporter"}

def reporter_node(state: AgentState):
    report = f"""MULTIMODAL AI RESEARCH REPORT 2026

Plan:
{state['plan']}

Key Research Findings:
{state['research_data']}

Summary:
{state['messages'][-1] if state['messages'] else 'No calculation done'}"""
    return {"final_report": report, "next": END}

print("LangGraph nodes defined")

LangGraph nodes defined


## 5. Build and Compile LangGraph

In [7]:
workflow = StateGraph(AgentState)

workflow.add_node("planner", planner_node)
workflow.add_node("researcher", researcher_node)
workflow.add_node("calculator", calculator_node)
workflow.add_node("reporter", reporter_node)

workflow.set_entry_point("planner")

workflow.add_conditional_edges(
    "planner",
    lambda x: x["next"],
    {"researcher": "researcher"}
)

workflow.add_conditional_edges(
    "researcher",
    lambda x: x["next"],
    {"calculator": "calculator"}
)

workflow.add_conditional_edges(
    "calculator",
    lambda x: x["next"],
    {"reporter": "reporter"}
)

workflow.add_conditional_edges(
    "reporter",
    lambda x: x["next"],
    {END: END}
)

graph = workflow.compile()

print("Multi-Agent LangGraph compiled")

Multi-Agent LangGraph compiled


## 6. Run the Full Multi-Agent System

In [8]:
def run_multi_agent_system(query: str):
    print(f"Starting Multi-Agent System for query:\n{query}\n")

    initial_state = {"messages": [query], "next": "planner"}

    result = graph.invoke(initial_state)

    print("\n" + "="*80)
    print("FINAL RESEARCH REPORT")
    print("="*80)
    print(result["final_report"])

    return result

# Run the demo
final_result = run_multi_agent_system(
    "Generate a short research report on the latest advancements in Multimodal AI models for 2026."
)

Starting Multi-Agent System for query:
Generate a short research report on the latest advancements in Multimodal AI models for 2026.


FINAL RESEARCH REPORT
MULTIMODAL AI RESEARCH REPORT 2026

Plan:
### 4-Step Plan for Generating a Research Report on Multimodal AI Models for 2026

**Step 1: Literature Review and Data Collection (Weeks 1-2)**
- Conduct a comprehensive review of recent academic papers, industry reports, and conference proceedings related to multimodal AI models.
- Identify key advancements, trends, and notable projects from 2023 to 2026.
- Gather data from reputable sources, including journals, tech blogs, and AI research organizations.

**Step 2: Analysis and Synthesis (Weeks 3-4)**
- Analyze the collected data to identify common themes, breakthroughs, and emerging technologies in multimodal AI.
- Synthesize findings into coherent sections, focusing on areas such as model architecture, applications, and performance improvements.

**Step 3: Drafting the Report (Weeks 5-6

## 7. Reflection Questions

1. How does LangGraph help with coordination compared to simple function calling?
2. What role does FAISS retrieval play in making the system more realistic?
3. How can we add execution tracking to this LangGraph workflow?
4. What improvements would you make to the communication between agents?

